# Scenario 6: Real-World ML: Handling Imbalance and Explaining Predictions

In our final scenario, we tackle two critical real-world topics: 
1.  **Class Imbalance:** What to do when one class is much rarer than another.
2.  **Model Interpretability:** How to explain the predictions of complex "black box" models.

We'll use a technique called SMOTE to handle imbalance and the powerful SHAP library to explain our model's predictions.

## 1. Import Libraries and Prepare Data

We'll need a few new libraries for this: `imblearn` for SMOTE and `shap` for interpretability. Make sure to install them: `pip install imbalanced-learn shap`

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
import shap
import matplotlib.pyplot as plt

# --- Data Prep --- 
column_names = [
    'id', 'clump_thickness', 'unif_cell_size', 'unif_cell_shape',
    'marg_adhesion', 'single_epith_cell_size', 'bare_nuclei',
    'bland_chrom', 'norm_nucleoli', 'mitoses', 'class'
]
df = pd.read_csv('../breast-cancer-wisconsin.data', names=column_names)
df['bare_nuclei'] = df['bare_nuclei'].replace('?', np.nan)
df['bare_nuclei'] = pd.to_numeric(df['bare_nuclei'])
df['bare_nuclei'].fillna(df['bare_nuclei'].median(), inplace=True)
df.drop('id', axis=1, inplace=True)
df['class'] = df['class'].map({2: 0, 4: 1})
X = df.drop('class', axis=1)
y = df['class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Part 1: Handling Class Imbalance with SMOTE

### 2. Check Class Distribution and Apply SMOTE

First, let's see the class imbalance in our training data. Then, we'll use SMOTE to create synthetic samples of the minority class (malignant cases).

In [ ]:
print("Class distribution before SMOTE:")
print(y_train.value_counts())

# HINT: Create an instance of SMOTE.
# Use the .fit_resample() method on the training data (X_train, y_train).
# IMPORTANT: Only apply SMOTE to the training data, never to the test data!

# YOUR CODE HERE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\nClass distribution after SMOTE:")
print(y_train_smote.value_counts())

### 3. Train a Model on the Balanced Data

Now, let's train an XGBoost model on our new, balanced training set and see how it performs on the original, imbalanced test set.

In [ ]:
# HINT: Create and fit an XGBClassifier, but this time use the `_smote` versions of the training data.

# YOUR CODE HERE
xgb_smote = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = xgb_smote.predict(X_test)

print("\nClassification Report (Model trained on SMOTE data):")
print(classification_report(y_test, y_pred_smote))

## Part 2: Model Interpretability with SHAP

### 4. Train a Model and Create a SHAP Explainer

For this part, we'll use a model trained on the original (non-SMOTE) data to see how SHAP explains its predictions. The first step is to train a model and then create a SHAP `TreeExplainer` for it.

In [ ]:
# Train a new model on the original training data
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train, y_train)

# HINT: Create a shap.TreeExplainer and pass the trained model to it.

# YOUR CODE HERE
explainer = shap.TreeExplainer(xgb_model)

### 5. Calculate and Visualize SHAP Values

Now we'll calculate the SHAP values for our test set. These values tell us how much each feature contributed to each prediction.

We will create two plots:
1.  A **force plot** for a single prediction to see the forces that push the model's output.
2.  A **summary plot** to see which features are most important globally and their impact.

In [ ]:
# HINT: Use explainer.shap_values() on the test set (X_test).

# YOUR CODE HERE
shap_values = explainer.shap_values(X_test)

# --- Visualize a single prediction --- 
print("Generating SHAP force plot for the first test instance...")
shap.initjs() # required for force plots in notebooks
# You may need to `display()` the plot if it doesn't render automatically
shap.force_plot(explainer.expected_value, shap_values[0,:], X_test.iloc[0,:])

In [ ]:
# --- Create a summary plot --- 
# HINT: Use shap.summary_plot() and pass in the shap_values and the test data (X_test).

# YOUR CODE HERE
print("\nGenerating SHAP summary plot...")
shap.summary_plot(shap_values, X_test)